In [1]:
import io
import numpy as np
import cv2
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from copy import deepcopy
import skimage
from skimage.io import imread
import scipy
import basicpy
import tifffile
from tqdm import tqdm
# from descartes import PolygonPatch
import geopandas as gpd
import basicpy

import sys
sys.path.append('/home/icb/lion.gleiter/projects/organoid_sam/SAM_with_Detection_Head')

from util.box_ops_numpy import mask_to_boxes, cxcywh_to_xyxy, xyxy_to_cxcywh, plot_boxes
from util import dataloading as dl
from util import postprocessing as pp
from util.samos import SAMOS

base_datadir = Path('/ictstr01/groups/shared/users/lion.gleiter/organoid_sam/original_data/')
results_dir = Path('/ictstr01/groups/shared/users/lion.gleiter/organoid_sam/results')

%load_ext autoreload
%autoreload 2

In [2]:
# samos = SAMOS(checkpoint_path='/ictstr01/groups/shared/users/lion.gleiter/organoid_sam/checkpoints_trained/DetectionHead_SAM_large_OrganoID_train_MultiOrg_train_macros_MultiOrg_train_normal_OrgaExtractor_train_OrgaQuant_train_OrgaSegment_train_Tellu_train_NewData_train_ablation_dataset_5_pre_OI_NeurIPS_True_32_200/DetectionHead_SAM_large_OrganoID_train_MultiOrg_train_macros_MultiOrg_train_normal_OrgaExtractor_train_OrgaQuant_train_OrgaSegment_train_Tellu_train_NewData_train_ablation_dataset_5_pre_OI_NeurIPS_True_32_200-best_epoch=449-val_loss=5.28.ckpt')

In [3]:
# last epoch
samos = SAMOS(checkpoint_path='/ictstr01/groups/shared/users/lion.gleiter/organoid_sam/checkpoints_trained/DetectionHead_SAM_large_OrganoID_train_MultiOrg_train_macros_MultiOrg_train_normal_OrgaExtractor_train_OrgaQuant_train_OrgaSegment_train_Tellu_train_NewData_train_ablation_dataset_5_pre_OI_NeurIPS_True_32_200/DetectionHead_SAM_large_OrganoID_train_MultiOrg_train_macros_MultiOrg_train_normal_OrgaExtractor_train_OrgaQuant_train_OrgaSegment_train_Tellu_train_NewData_train_ablation_dataset_5_pre_OI_NeurIPS_True_32_200-last_epoch=499-val_loss=5.44.ckpt')

/home/icb/lion.gleiter/projects/organoid_sam/segment-anything/segment-anything/segment_anything/build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = 

In [4]:
# samos = SAMOS(checkpoint_path='/ictstr01/groups/shared/users/lion.gleiter/organoid_sam/checkpoints_trained/DetectionHead_SAM_large_OrganoID_train_MultiOrg_train_macros_MultiOrg_train_normal_OrgaExtractor_train_OrgaQuant_train_OrgaSegment_train_Tellu_train_NewData_train_added_eval_True_32_200/DetectionHead_SAM_large_OrganoID_train_MultiOrg_train_macros_MultiOrg_train_normal_OrgaExtractor_train_OrgaQuant_train_OrgaSegment_train_Tellu_train_NewData_train_added_eval_True_32_200-best_epoch=649-val_loss=4.83.ckpt')

In [5]:
# retrained last epoch
# samos = SAMOS(checkpoint_path='/ictstr01/groups/shared/users/lion.gleiter/organoid_sam/checkpoints_trained/DetectionHead_SAM_large_OrganoID_train_MultiOrg_train_macros_MultiOrg_train_normal_OrgaExtractor_train_OrgaQuant_train_OrgaSegment_train_Tellu_train_NewData_train_added_eval_True_32_200/DetectionHead_SAM_large_OrganoID_train_MultiOrg_train_macros_MultiOrg_train_normal_OrgaExtractor_train_OrgaQuant_train_OrgaSegment_train_Tellu_train_NewData_train_added_eval_True_32_200-last_epoch=799-val_loss=4.93.ckpt')

In [6]:
def show_points(coords, labels, ax, marker_size=375):
    pos_points = coords[labels==1]
    neg_points = coords[labels==0]
    ax.scatter(pos_points[:, 0], pos_points[:, 1], color='green', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)
    ax.scatter(neg_points[:, 0], neg_points[:, 1], color='red', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)   
    
def show_box(box, ax, color='red'):
    y_min, x_min, y_max, x_max = box
    
    # Calculate width and height of the box
    width = x_max - x_min
    height = y_max - y_min

    ax.add_patch(plt.Rectangle((x_min, y_min), width, height, edgecolor=color, facecolor=(0,0,0,0), lw=2))

def show_mask(contour, ax, random_color=False):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.4])], axis=0)
    else:
        color = np.array([30/255, 184/255, 255/255, 0.4])

    contour = gpd.GeoSeries(contour)
    contour.plot(color=color, ax=ax)

def visualize(im, contours, boxes, random_color=True, box_color='red', format='yxyx_px', ax=None, **fig_kwargs):
    # Format boxes
    H, W = im.shape[:2]

    if format=='cycxhw_01':
        # boxes are in cycxhw format normalized by the longest image side. Converts to min/max coordinates in pixels.
        original_boxes = cxcywh_to_xyxy(boxes) * max(H, W)
    elif format=='cycxhw_px':
        # boxes are in cycxhw format in pixels. Converts to min/max coordinates in pixels.
        original_boxes = cxcywh_to_xyxy(boxes)
    elif format=='yxyx_01':
        # boxes are in min/max coordinates normalized by the longest image side. Converts to min/max coordinates in pixels.
        original_boxes = boxes * max(H, W)
    elif format=='yxyx_px':
        # boxes are in min/max coordinates in pixels. No conversion necessary.
        original_boxes = boxes
    else:
        raise ValueError(format)

    if ax is None:
        fig, ax = plt.subplots(1, 1, **fig_kwargs)
    
    ax.imshow(im)
    ax.axis(False)
    for i in range(original_boxes.shape[0]):
        show_mask(contours[i], ax, random_color=random_color)
        show_box(original_boxes[i], ax, color=box_color)

## OrganoID

In [7]:
ds = dl.OrganoID(split='test')
pq_data = []
for idx in range(len(ds)):
    im, gt_mask, gt_boxes, im_path, im_ID = ds[idx]
    im, flatfield = dl.normalize(im)

    H, W = im.shape[:2]
    patch_size = int(np.ceil(max(H, W) * 7 / 12))
    print(patch_size)

    contours, boxes, scores = samos.forward(im, patch_size=patch_size, predict_masks=True)
    for thres in [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95]:
        contours, boxes, scores = samos.set_threshold(conf_thres=thres, predict_masks=True)
        if contours is not None:
            iou_matrix = pp.compute_iou_matrix_segmentation_contours(contours, pp.convert_mask_to_binary(gt_mask).astype(np.uint8))
            tp, fp, fn, pq, precision, recall, f1_score, mean_iou, dice = \
                pp.compute_metrics_segmentation_from_iou_matrix(iou_matrix=iou_matrix)
        else:
            iou_matrix = pp.compute_iou_matrix_detection(boxes, gt_boxes)
            tp, fp, fn, pq, precision, recall, f1_score, mean_iou, dice = \
                pp.compute_metrics_detection_from_iou_matrix(iou_matrix=iou_matrix)

        print("Threshold", thres)
        print("tp, fp, fn, pq, precision, recall, f1_score, mean_iou:", tp, fp, fn, pq, precision, recall, f1_score, mean_iou)
        pq_data.append((f'{thres:4.2f}', pq, f1_score, precision, recall, mean_iou))


        fig, ax = plt.subplots(1, 1, figsize=(12*4, 12*4), dpi=200)
        plot_boxes(im, gt_boxes, format='yxyx_px', ax=ax, color='blue')
        plot_boxes(im, boxes, format='yxyx_px', ax=ax, show_image=False, color='red')
        plot_dir = results_dir / 'plots' / 'MultiOrg_test_added_eval'
        plot_dir.mkdir(exist_ok=True)
        plt.savefig(plot_dir / f'{str(ds)}_{ds.split}_{idx}_thres_{int(thres*100)}.png', dpi=200)
        plt.close('all')

    # fig, ax = plt.subplots(1, 1, figsize=(10, 10), dpi=80)
    # plot_boxes(im, gt_boxes, format='yxyx_px', ax=ax, color='blue')
    # plot_boxes(im, boxes, format='yxyx_px', ax=ax, show_image=False, color='red')

pq_data = pd.DataFrame(data=pq_data, columns=["thres", "pq", "f1_score", "precision", "recall", "iou"])
pq_data.to_csv(results_dir / f'test_metrics_added_eval_{str(ds)}_{ds.split}.csv', index=False)

pq_data.groupby('thres').mean()
# plot_boxes(im, boxes, format='yxyx_px')

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


976
Threshold 0.5
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 64 107 116 0.2758711760212687 0.3742690058479532 0.35555555555555557 0.36467236467236464 0.7564904904958228
Threshold 0.6
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 61 89 119 0.2815202601479259 0.4066666666666667 0.3388888888888889 0.36969696969696975 0.76148922826898
Threshold 0.7
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 59 71 121 0.2911250693925507 0.45384615384615384 0.3277777777777778 0.3806451612903226 0.7648200975567009
Threshold 0.8
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 55 52 125 0.2994225568747437 0.514018691588785 0.3055555555555556 0.3832752613240418 0.7812206711186493
Threshold 0.85
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 50 44 130 0.2875695909761604 0.5319148936170213 0.2777777777777778 0.36496350364963503 0.7879406792746795
Threshold 0.9
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 43 36 137 0.26428264165667786 0.5443037974683544 0.23

,pq,f1_score,precision,recall,iou
thres,,,,,
0.50,0.487192,0.597299,0.491087,0.824000,0.811195
0.60,0.515346,0.631170,0.538544,0.815637,0.812273
0.70,0.539043,0.659505,0.588395,0.803088,0.813628
0.80,0.558661,0.679730,0.636142,0.775891,0.818681
0.85,0.566661,0.686497,0.666754,0.754057,0.821975
0.90,0.580986,0.704416,0.720046,0.728205,0.821712
0.95,0.582469,0.704343,0.825306,0.651209,0.825612


In [ ]:
ds = dl.OrganoID(split='test_mouse')
pq_data = []
for idx in range(len(ds)):
    im, gt_mask, gt_boxes, im_path, im_ID = ds[idx]
    im, flatfield = dl.normalize(im)

    H, W = im.shape[:2]
    patch_size = int(np.ceil(max(H, W) * 7 / 12))
    print(patch_size)

    contours, boxes, scores = samos.forward(im, patch_size=patch_size, predict_masks=True)
    for thres in [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95]:
        contours, boxes, scores = samos.set_threshold(conf_thres=thres, predict_masks=True)
        if contours is not None:
            iou_matrix = pp.compute_iou_matrix_segmentation_contours(contours, pp.convert_mask_to_binary(gt_mask).astype(np.uint8))
            tp, fp, fn, pq, precision, recall, f1_score, mean_iou, dice = \
                pp.compute_metrics_segmentation_from_iou_matrix(iou_matrix=iou_matrix)
        else:
            iou_matrix = pp.compute_iou_matrix_detection(boxes, gt_boxes)
            tp, fp, fn, pq, precision, recall, f1_score, mean_iou, dice = \
                pp.compute_metrics_detection_from_iou_matrix(iou_matrix=iou_matrix)

        print("Threshold", thres)
        print("tp, fp, fn, pq, precision, recall, f1_score, mean_iou:", tp, fp, fn, pq, precision, recall, f1_score, mean_iou)
        pq_data.append((f'{thres:4.2f}', pq, f1_score, precision, recall, mean_iou))


        fig, ax = plt.subplots(1, 1, figsize=(12*4, 12*4), dpi=200)
        plot_boxes(im, gt_boxes, format='yxyx_px', ax=ax, color='blue')
        plot_boxes(im, boxes, format='yxyx_px', ax=ax, show_image=False, color='red')
        plot_dir = results_dir / 'plots' / 'MultiOrg_test_added_eval'
        plot_dir.mkdir(exist_ok=True)
        plt.savefig(plot_dir / f'{str(ds)}_{ds.split}_{idx}_thres_{int(thres*100)}.png', dpi=200)
        plt.close('all')

    # fig, ax = plt.subplots(1, 1, figsize=(10, 10), dpi=80)
    # plot_boxes(im, gt_boxes, format='yxyx_px', ax=ax, color='blue')
    # plot_boxes(im, boxes, format='yxyx_px', ax=ax, show_image=False, color='red')

pq_data = pd.DataFrame(data=pq_data, columns=["thres", "pq", "f1_score", "precision", "recall", "iou"])
pq_data.to_csv(results_dir / f'test_metrics_added_eval_{str(ds)}_{ds.split}.csv', index=False)

pq_data.groupby('thres').mean()
# plot_boxes(im, boxes, format='yxyx_px')

1512
Threshold 0.5
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 10 11 2 0.49068046436435847 0.47619047619047616 0.8333333333333334 0.6060606060606061 0.8096227662011914
Threshold 0.6
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 10 9 2 0.5223372685168977 0.5263157894736842 0.8333333333333334 0.6451612903225806 0.8096227662011914
Threshold 0.7
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 10 8 2 0.5397485108007943 0.5555555555555556 0.8333333333333334 0.6666666666666667 0.8096227662011914
Threshold 0.8
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 10 6 2 0.5783019758579939 0.625 0.8333333333333334 0.7142857142857143 0.8096227662011914
Threshold 0.85
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 10 5 2 0.5997205675564381 0.6666666666666666 0.8333333333333334 0.7407407407407408 0.8096227662011914
Threshold 0.9
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 10 1 2 0.7040197966966882 0.9090909090909091 0.8333333333333334 0.86956521739130

,pq,f1_score,precision,recall,iou
thres,,,,,
0.50,0.613804,0.714344,0.631884,0.864687,0.857635
0.60,0.632521,0.736424,0.675379,0.851493,0.856830
0.70,0.648156,0.754952,0.725676,0.830293,0.855713
0.80,0.652426,0.758483,0.759352,0.809615,0.859210
0.85,0.655524,0.761954,0.772825,0.791326,0.859055
0.90,0.655589,0.762292,0.821940,0.749542,0.858484
0.95,0.625124,0.720544,0.858036,0.659400,0.867173


: 

In [7]:
ds = dl.OrganoID(split='test_only_mouse')
pq_data = []
for idx in range(len(ds)):
    im, gt_mask, gt_boxes, im_path, im_ID = ds[idx]
    im, flatfield = dl.normalize(im)

    H, W = im.shape[:2]
    patch_size = int(np.ceil(max(H, W) * 7 / 12))
    print(patch_size)

    contours, boxes, scores = samos.forward(im, patch_size=patch_size, predict_masks=True)
    for thres in [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95]:
        contours, boxes, scores = samos.set_threshold(conf_thres=thres, predict_masks=True)
        if contours is not None:
            iou_matrix = pp.compute_iou_matrix_segmentation_contours(contours, pp.convert_mask_to_binary(gt_mask).astype(np.uint8))
            tp, fp, fn, pq, precision, recall, f1_score, mean_iou, dice = \
                pp.compute_metrics_segmentation_from_iou_matrix(iou_matrix=iou_matrix)
        else:
            iou_matrix = pp.compute_iou_matrix_detection(boxes, gt_boxes)
            tp, fp, fn, pq, precision, recall, f1_score, mean_iou, dice = \
                pp.compute_metrics_detection_from_iou_matrix(iou_matrix=iou_matrix)

        print("Threshold", thres)
        print("tp, fp, fn, pq, precision, recall, f1_score, mean_iou:", tp, fp, fn, pq, precision, recall, f1_score, mean_iou)
        pq_data.append((f'{thres:4.2f}', pq, f1_score, precision, recall, mean_iou))


        fig, ax = plt.subplots(1, 1, figsize=(12*4, 12*4), dpi=200)
        plot_boxes(im, gt_boxes, format='yxyx_px', ax=ax, color='blue')
        plot_boxes(im, boxes, format='yxyx_px', ax=ax, show_image=False, color='red')
        plot_dir = results_dir / 'plots' / 'MultiOrg_test_added_eval'
        plot_dir.mkdir(exist_ok=True)
        plt.savefig(plot_dir / f'{str(ds)}_{ds.split}_{idx}_thres_{int(thres*100)}.png', dpi=200)
        plt.close('all')

    # fig, ax = plt.subplots(1, 1, figsize=(10, 10), dpi=80)
    # plot_boxes(im, gt_boxes, format='yxyx_px', ax=ax, color='blue')
    # plot_boxes(im, boxes, format='yxyx_px', ax=ax, show_image=False, color='red')

pq_data = pd.DataFrame(data=pq_data, columns=["thres", "pq", "f1_score", "precision", "recall", "iou"])
pq_data.to_csv(results_dir / f'test_metrics_added_eval_{str(ds)}_{ds.split}.csv', index=False)

pq_data.groupby('thres').mean()
# plot_boxes(im, boxes, format='yxyx_px')

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


1512
Threshold 0.5
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 11 10 1 0.5094869334035363 0.5238095238095238 0.9166666666666666 0.6666666666666667 0.7642304001053045
Threshold 0.6
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 11 9 1 0.5254084000723969 0.55 0.9166666666666666 0.6874999999999999 0.7642304001053045
Threshold 0.7
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 11 7 1 0.56043562674389 0.6111111111111112 0.9166666666666666 0.7333333333333334 0.7642304001053045
Threshold 0.8
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 10 5 2 0.5838820189922579 0.6666666666666666 0.8333333333333334 0.7407407407407408 0.7882407256395482
Threshold 0.85
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 10 5 2 0.5838820189922579 0.6666666666666666 0.8333333333333334 0.7407407407407408 0.7882407256395482
Threshold 0.9
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 10 2 2 0.6568672713662901 0.8333333333333334 0.8333333333333334 0.8333333333333334 0.

,pq,f1_score,precision,recall,iou
thres,,,,,
0.50,0.597067,0.713031,0.588985,0.907012,0.834915
0.60,0.622626,0.742006,0.633145,0.900915,0.836689
0.70,0.642726,0.766487,0.675533,0.888720,0.836345
0.80,0.664844,0.786840,0.747391,0.832656,0.842527
0.85,0.654926,0.775245,0.764860,0.794377,0.842842
0.90,0.669154,0.792131,0.827288,0.765244,0.844460
0.95,0.609209,0.712569,0.854952,0.625339,0.852925


In [8]:
ds = dl.OrganoID(split='test_ACC')
pq_data = []
for idx in range(len(ds)):
    im, gt_mask, gt_boxes, im_path, im_ID = ds[idx]
    im, flatfield = dl.normalize(im)

    H, W = im.shape[:2]
    patch_size = int(np.ceil(max(H, W) * 7 / 12))
    print(patch_size)

    contours, boxes, scores = samos.forward(im, patch_size=patch_size, predict_masks=True)
    for thres in [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95]:
        contours, boxes, scores = samos.set_threshold(conf_thres=thres, predict_masks=True)
        if contours is not None:
            iou_matrix = pp.compute_iou_matrix_segmentation_contours(contours, pp.convert_mask_to_binary(gt_mask).astype(np.uint8))
            tp, fp, fn, pq, precision, recall, f1_score, mean_iou, dice = \
                pp.compute_metrics_segmentation_from_iou_matrix(iou_matrix=iou_matrix)
        else:
            iou_matrix = pp.compute_iou_matrix_detection(boxes, gt_boxes)
            tp, fp, fn, pq, precision, recall, f1_score, mean_iou, dice = \
                pp.compute_metrics_detection_from_iou_matrix(iou_matrix=iou_matrix)

        print("Threshold", thres)
        print("tp, fp, fn, pq, precision, recall, f1_score, mean_iou:", tp, fp, fn, pq, precision, recall, f1_score, mean_iou)
        pq_data.append((f'{thres:4.2f}', pq, f1_score, precision, recall, mean_iou))


        fig, ax = plt.subplots(1, 1, figsize=(12*4, 12*4), dpi=200)
        plot_boxes(im, gt_boxes, format='yxyx_px', ax=ax, color='blue')
        plot_boxes(im, boxes, format='yxyx_px', ax=ax, show_image=False, color='red')
        plot_dir = results_dir / 'plots' / 'MultiOrg_test_added_eval'
        plot_dir.mkdir(exist_ok=True)
        plt.savefig(plot_dir / f'{str(ds)}_{ds.split}_{idx}_thres_{int(thres*100)}.png', dpi=200)
        plt.close('all')

    # fig, ax = plt.subplots(1, 1, figsize=(10, 10), dpi=80)
    # plot_boxes(im, gt_boxes, format='yxyx_px', ax=ax, color='blue')
    # plot_boxes(im, boxes, format='yxyx_px', ax=ax, show_image=False, color='red')

pq_data = pd.DataFrame(data=pq_data, columns=["thres", "pq", "f1_score", "precision", "recall", "iou"])
pq_data.to_csv(results_dir / f'test_metrics_added_eval_{str(ds)}_{ds.split}.csv', index=False)

pq_data.groupby('thres').mean()
# plot_boxes(im, boxes, format='yxyx_px')

438
Threshold 0.5
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 9 5 0 0.6340950487826639 0.6428571428571429 1.0 0.782608695652174 0.8102325623334039
Threshold 0.6
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 9 4 0 0.6629175510000578 0.6923076923076923 1.0 0.8181818181818181 0.8102325623334039
Threshold 0.7
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 9 2 0 0.7292093061000635 0.8181818181818182 1.0 0.9 0.8102325623334039
Threshold 0.8
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 9 2 0 0.7292093061000635 0.8181818181818182 1.0 0.9 0.8102325623334039
Threshold 0.85
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 8 1 1 0.7174275571196396 0.8888888888888888 0.8888888888888888 0.8888888888888888 0.8071060017595946
Threshold 0.9
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 8 1 1 0.7174275571196396 0.8888888888888888 0.8888888888888888 0.8888888888888888 0.8071060017595946
Threshold 0.95
tp, fp, fn, pq, precision, recall, f1_score, mean_i

,pq,f1_score,precision,recall,iou
thres,,,,,
0.50,0.565008,0.666520,0.550206,0.874805,0.847518
0.60,0.581816,0.686401,0.586776,0.863731,0.847857
0.70,0.591182,0.697624,0.626281,0.846483,0.848380
0.80,0.607901,0.715398,0.677952,0.824774,0.850178
0.85,0.607587,0.713025,0.721164,0.781746,0.851725
0.90,0.615196,0.721077,0.776720,0.755965,0.850317
0.95,0.626462,0.720874,0.924107,0.661453,0.861864


In [9]:
ds = dl.OrganoID(split='test_Lung')
pq_data = []
for idx in range(len(ds)):
    im, gt_mask, gt_boxes, im_path, im_ID = ds[idx]
    im, flatfield = dl.normalize(im)

    H, W = im.shape[:2]
    patch_size = int(np.ceil(max(H, W) * 7 / 12))
    print(patch_size)

    contours, boxes, scores = samos.forward(im, patch_size=patch_size, predict_masks=True)
    for thres in [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95]:
        contours, boxes, scores = samos.set_threshold(conf_thres=thres, predict_masks=True)
        if contours is not None:
            iou_matrix = pp.compute_iou_matrix_segmentation_contours(contours, pp.convert_mask_to_binary(gt_mask).astype(np.uint8))
            tp, fp, fn, pq, precision, recall, f1_score, mean_iou, dice = \
                pp.compute_metrics_segmentation_from_iou_matrix(iou_matrix=iou_matrix)
        else:
            iou_matrix = pp.compute_iou_matrix_detection(boxes, gt_boxes)
            tp, fp, fn, pq, precision, recall, f1_score, mean_iou, dice = \
                pp.compute_metrics_detection_from_iou_matrix(iou_matrix=iou_matrix)

        print("Threshold", thres)
        print("tp, fp, fn, pq, precision, recall, f1_score, mean_iou:", tp, fp, fn, pq, precision, recall, f1_score, mean_iou)
        pq_data.append((f'{thres:4.2f}', pq, f1_score, precision, recall, mean_iou))


        fig, ax = plt.subplots(1, 1, figsize=(12*4, 12*4), dpi=200)
        plot_boxes(im, gt_boxes, format='yxyx_px', ax=ax, color='blue')
        plot_boxes(im, boxes, format='yxyx_px', ax=ax, show_image=False, color='red')
        plot_dir = results_dir / 'plots' / 'MultiOrg_test_added_eval'
        plot_dir.mkdir(exist_ok=True)
        plt.savefig(plot_dir / f'{str(ds)}_{ds.split}_{idx}_thres_{int(thres*100)}.png', dpi=200)
        plt.close('all')

    # fig, ax = plt.subplots(1, 1, figsize=(10, 10), dpi=80)
    # plot_boxes(im, gt_boxes, format='yxyx_px', ax=ax, color='blue')
    # plot_boxes(im, boxes, format='yxyx_px', ax=ax, show_image=False, color='red')

pq_data = pd.DataFrame(data=pq_data, columns=["thres", "pq", "f1_score", "precision", "recall", "iou"])
pq_data.to_csv(results_dir / f'test_metrics_added_eval_{str(ds)}_{ds.split}.csv', index=False)

pq_data.groupby('thres').mean()
# plot_boxes(im, boxes, format='yxyx_px')

598
Threshold 0.5
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 22 8 1 0.756671037723776 0.7333333333333333 0.9565217391304348 0.8301886792452831 0.9114446590763664
Threshold 0.6
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 22 8 1 0.756671037723776 0.7333333333333333 0.9565217391304348 0.8301886792452831 0.9114446590763664
Threshold 0.7
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 22 8 1 0.756671037723776 0.7333333333333333 0.9565217391304348 0.8301886792452831 0.9114446590763664
Threshold 0.8
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 22 8 1 0.756671037723776 0.7333333333333333 0.9565217391304348 0.8301886792452831 0.9114446590763664
Threshold 0.85
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 21 8 2 0.7343717363590881 0.7241379310344828 0.9130434782608695 0.8076923076923076 0.9092221497779187
Threshold 0.9
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 21 8 2 0.7343717363590881 0.7241379310344828 0.9130434782608695 0.807692307

,pq,f1_score,precision,recall,iou
thres,,,,,
0.50,0.740090,0.825986,0.717953,0.985304,0.893917
0.60,0.750921,0.836635,0.738568,0.975780,0.895822
0.70,0.758482,0.845139,0.752091,0.975780,0.895822
0.80,0.774503,0.863416,0.779188,0.975780,0.895831
0.85,0.778493,0.868405,0.791351,0.968533,0.895460
0.90,0.790187,0.882170,0.821824,0.956765,0.895324
0.95,0.809347,0.899890,0.886417,0.917927,0.900023


In [10]:
ds = dl.OrganoID(split='test_C')
pq_data = []
for idx in range(len(ds)):
    im, gt_mask, gt_boxes, im_path, im_ID = ds[idx]
    im, flatfield = dl.normalize(im)

    H, W = im.shape[:2]
    patch_size = int(np.ceil(max(H, W) * 7 / 12))
    print(patch_size)

    contours, boxes, scores = samos.forward(im, patch_size=patch_size, predict_masks=True)
    for thres in [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95]:
        contours, boxes, scores = samos.set_threshold(conf_thres=thres, predict_masks=True)
        if contours is not None:
            iou_matrix = pp.compute_iou_matrix_segmentation_contours(contours, pp.convert_mask_to_binary(gt_mask).astype(np.uint8))
            tp, fp, fn, pq, precision, recall, f1_score, mean_iou, dice = \
                pp.compute_metrics_segmentation_from_iou_matrix(iou_matrix=iou_matrix)
        else:
            iou_matrix = pp.compute_iou_matrix_detection(boxes, gt_boxes)
            tp, fp, fn, pq, precision, recall, f1_score, mean_iou, dice = \
                pp.compute_metrics_detection_from_iou_matrix(iou_matrix=iou_matrix)

        print("Threshold", thres)
        print("tp, fp, fn, pq, precision, recall, f1_score, mean_iou:", tp, fp, fn, pq, precision, recall, f1_score, mean_iou)
        pq_data.append((f'{thres:4.2f}', pq, f1_score, precision, recall, mean_iou))


        fig, ax = plt.subplots(1, 1, figsize=(12*4, 12*4), dpi=200)
        plot_boxes(im, gt_boxes, format='yxyx_px', ax=ax, color='blue')
        plot_boxes(im, boxes, format='yxyx_px', ax=ax, show_image=False, color='red')
        plot_dir = results_dir / 'plots' / 'MultiOrg_test_added_eval'
        plot_dir.mkdir(exist_ok=True)
        plt.savefig(plot_dir / f'{str(ds)}_{ds.split}_{idx}_thres_{int(thres*100)}.png', dpi=200)
        plt.close('all')

    # fig, ax = plt.subplots(1, 1, figsize=(10, 10), dpi=80)
    # plot_boxes(im, gt_boxes, format='yxyx_px', ax=ax, color='blue')
    # plot_boxes(im, boxes, format='yxyx_px', ax=ax, show_image=False, color='red')

pq_data = pd.DataFrame(data=pq_data, columns=["thres", "pq", "f1_score", "precision", "recall", "iou"])
pq_data.to_csv(results_dir / f'test_metrics_added_eval_{str(ds)}_{ds.split}.csv', index=False)

pq_data.groupby('thres').mean()
# plot_boxes(im, boxes, format='yxyx_px')

598
Threshold 0.5
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 16 22 4 0.47320248346730276 0.42105263157894735 0.8 0.5517241379310345 0.8576795012844862
Threshold 0.6
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 16 19 4 0.4990135280200647 0.45714285714285713 0.8 0.5818181818181818 0.8576795012844862
Threshold 0.7
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 16 17 4 0.5178442271906332 0.48484848484848486 0.8 0.6037735849056605 0.8576795012844862
Threshold 0.8
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 16 15 4 0.538151843943207 0.5161290322580645 0.8 0.6274509803921569 0.8576795012844862
Threshold 0.85
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 16 12 4 0.5717863341896575 0.5714285714285714 0.8 0.6666666666666666 0.8576795012844862
Threshold 0.9
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 14 10 6 0.5552674476706448 0.5833333333333334 0.7 0.6363636363636365 0.8725631320538705
Threshold 0.95
tp, fp, fn, pq, precision, recall, 

,pq,f1_score,precision,recall,iou
thres,,,,,
0.50,0.450539,0.527023,0.378537,0.912179,0.852088
0.60,0.474330,0.553868,0.409615,0.895513,0.853545
0.70,0.522649,0.611641,0.470157,0.895513,0.852400
0.80,0.560167,0.654144,0.527796,0.878686,0.855324
0.85,0.584623,0.683123,0.571768,0.868269,0.854972
0.90,0.615387,0.716994,0.631798,0.841186,0.857990
0.95,0.664175,0.771418,0.739494,0.824359,0.861287


## OrgaSegment

In [ ]:
ds = dl.OrgaSegment(split='test')
pq_data = []
for idx in range(len(ds)):
    im, gt_mask, gt_boxes, im_path, im_ID = ds[idx]
    im, flatfield = dl.normalize(im)

    H, W = im.shape[:2]
    # patch_size = int(np.ceil(max(H, W) * 7 / 12))
    patch_size = 512
    print(patch_size)

    contours, boxes, scores = samos.forward(im, patch_size=patch_size, predict_masks=True)
    for thres in [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95]:
        contours, boxes, scores = samos.set_threshold(conf_thres=thres, predict_masks=True)
        if contours is not None:
            iou_matrix = pp.compute_iou_matrix_segmentation_contours(contours, pp.convert_mask_to_binary(gt_mask).astype(np.uint8))
            tp, fp, fn, pq, precision, recall, f1_score, mean_iou, dice = \
                pp.compute_metrics_segmentation_from_iou_matrix(iou_matrix=iou_matrix)
        else:
            iou_matrix = pp.compute_iou_matrix_detection(boxes, gt_boxes)
            tp, fp, fn, pq, precision, recall, f1_score, mean_iou, dice = \
                pp.compute_metrics_detection_from_iou_matrix(iou_matrix=iou_matrix)

        print("Threshold", thres)
        print("tp, fp, fn, pq, precision, recall, f1_score, mean_iou:", tp, fp, fn, pq, precision, recall, f1_score, mean_iou)
        pq_data.append((f'{thres:4.2f}', pq, f1_score, precision, recall, mean_iou))


        fig, ax = plt.subplots(1, 1, figsize=(12*4, 12*4), dpi=200)
        plot_boxes(im, gt_boxes, format='yxyx_px', ax=ax, color='blue')
        plot_boxes(im, boxes, format='yxyx_px', ax=ax, show_image=False, color='red')
        plot_dir = results_dir / 'plots' / 'MultiOrg_test_added_eval'
        plot_dir.mkdir(exist_ok=True)
        plt.savefig(plot_dir / f'{str(ds)}_{ds.split}_{idx}_thres_{int(thres*100)}.png', dpi=200)
        plt.close('all')

    # fig, ax = plt.subplots(1, 1, figsize=(10, 10), dpi=80)
    # plot_boxes(im, gt_boxes, format='yxyx_px', ax=ax, color='blue')
    # plot_boxes(im, boxes, format='yxyx_px', ax=ax, show_image=False, color='red')

pq_data = pd.DataFrame(data=pq_data, columns=["thres", "pq", "f1_score", "precision", "recall", "iou"])
pq_data.to_csv(results_dir / f'test_metrics_added_eval_{str(ds)}_{ds.split}.csv', index=False)

pq_data.groupby('thres').mean()
# plot_boxes(im, boxes, format='yxyx_px')

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


976
Threshold 0.5
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 64 107 116 0.2758711760212687 0.3742690058479532 0.35555555555555557 0.36467236467236464 0.7564904904958228
Threshold 0.6
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 61 89 119 0.2815202601479259 0.4066666666666667 0.3388888888888889 0.36969696969696975 0.76148922826898
Threshold 0.7
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 59 71 121 0.2911250693925507 0.45384615384615384 0.3277777777777778 0.3806451612903226 0.7648200975567009
Threshold 0.8
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 55 52 125 0.2994225568747437 0.514018691588785 0.3055555555555556 0.3832752613240418 0.7812206711186493
Threshold 0.85
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 50 44 130 0.2875695909761604 0.5319148936170213 0.2777777777777778 0.36496350364963503 0.7879406792746795
Threshold 0.9
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 43 36 137 0.26428264165667786 0.5443037974683544 0.23

,pq,f1_score,precision,recall,iou
thres,,,,,
0.50,0.487192,0.597299,0.491087,0.824000,0.811195
0.60,0.515346,0.631170,0.538544,0.815637,0.812273
0.70,0.539043,0.659505,0.588395,0.803088,0.813628
0.80,0.558661,0.679730,0.636142,0.775891,0.818681
0.85,0.566661,0.686497,0.666754,0.754057,0.821975
0.90,0.580986,0.704416,0.720046,0.728205,0.821712
0.95,0.582469,0.704343,0.825306,0.651209,0.825612


## OrgaQuant

In [ ]:
ds = dl.OrgaQuant(split='test')
pq_data = []
for idx in range(len(ds)):
    im, gt_mask, gt_boxes, im_path, im_ID = ds[idx]
    im, flatfield = dl.normalize(im)

    H, W = im.shape[:2]
    # patch_size = int(np.ceil(max(H, W) * 7 / 12))
    patch_size = 512
    print(patch_size)

    contours, boxes, scores = samos.forward(im, patch_size=patch_size, predict_masks=True)
    for thres in [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95]:
        contours, boxes, scores = samos.set_threshold(conf_thres=thres, predict_masks=True)
        if contours is not None:
            iou_matrix = pp.compute_iou_matrix_segmentation_contours(contours, pp.convert_mask_to_binary(gt_mask).astype(np.uint8))
            tp, fp, fn, pq, precision, recall, f1_score, mean_iou, dice = \
                pp.compute_metrics_segmentation_from_iou_matrix(iou_matrix=iou_matrix)
        else:
            iou_matrix = pp.compute_iou_matrix_detection(boxes, gt_boxes)
            tp, fp, fn, pq, precision, recall, f1_score, mean_iou, dice = \
                pp.compute_metrics_detection_from_iou_matrix(iou_matrix=iou_matrix)

        print("Threshold", thres)
        print("tp, fp, fn, pq, precision, recall, f1_score, mean_iou:", tp, fp, fn, pq, precision, recall, f1_score, mean_iou)
        pq_data.append((f'{thres:4.2f}', pq, f1_score, precision, recall, mean_iou))


        fig, ax = plt.subplots(1, 1, figsize=(12*4, 12*4), dpi=200)
        plot_boxes(im, gt_boxes, format='yxyx_px', ax=ax, color='blue')
        plot_boxes(im, boxes, format='yxyx_px', ax=ax, show_image=False, color='red')
        plot_dir = results_dir / 'plots' / 'MultiOrg_test_added_eval'
        plot_dir.mkdir(exist_ok=True)
        plt.savefig(plot_dir / f'{str(ds)}_{ds.split}_{idx}_thres_{int(thres*100)}.png', dpi=200)
        plt.close('all')

    # fig, ax = plt.subplots(1, 1, figsize=(10, 10), dpi=80)
    # plot_boxes(im, gt_boxes, format='yxyx_px', ax=ax, color='blue')
    # plot_boxes(im, boxes, format='yxyx_px', ax=ax, show_image=False, color='red')

pq_data = pd.DataFrame(data=pq_data, columns=["thres", "pq", "f1_score", "precision", "recall", "iou"])
pq_data.to_csv(results_dir / f'test_metrics_added_eval_{str(ds)}_{ds.split}.csv', index=False)

pq_data.groupby('thres').mean()
# plot_boxes(im, boxes, format='yxyx_px')

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


976
Threshold 0.5
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 64 107 116 0.2758711760212687 0.3742690058479532 0.35555555555555557 0.36467236467236464 0.7564904904958228
Threshold 0.6
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 61 89 119 0.2815202601479259 0.4066666666666667 0.3388888888888889 0.36969696969696975 0.76148922826898
Threshold 0.7
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 59 71 121 0.2911250693925507 0.45384615384615384 0.3277777777777778 0.3806451612903226 0.7648200975567009
Threshold 0.8
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 55 52 125 0.2994225568747437 0.514018691588785 0.3055555555555556 0.3832752613240418 0.7812206711186493
Threshold 0.85
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 50 44 130 0.2875695909761604 0.5319148936170213 0.2777777777777778 0.36496350364963503 0.7879406792746795
Threshold 0.9
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 43 36 137 0.26428264165667786 0.5443037974683544 0.23

,pq,f1_score,precision,recall,iou
thres,,,,,
0.50,0.487192,0.597299,0.491087,0.824000,0.811195
0.60,0.515346,0.631170,0.538544,0.815637,0.812273
0.70,0.539043,0.659505,0.588395,0.803088,0.813628
0.80,0.558661,0.679730,0.636142,0.775891,0.818681
0.85,0.566661,0.686497,0.666754,0.754057,0.821975
0.90,0.580986,0.704416,0.720046,0.728205,0.821712
0.95,0.582469,0.704343,0.825306,0.651209,0.825612


## OrgaExtractor

In [ ]:
ds = dl.OrgaExtractor(split='test')
pq_data = []
for idx in range(len(ds)):
    im, gt_mask, gt_boxes, im_path, im_ID = ds[idx]
    im, flatfield = dl.normalize(im)

    H, W = im.shape[:2]
    # patch_size = int(np.ceil(max(H, W) * 7 / 12))
    patch_size = 512
    print(patch_size)

    contours, boxes, scores = samos.forward(im, patch_size=patch_size, predict_masks=True)
    for thres in [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95]:
        contours, boxes, scores = samos.set_threshold(conf_thres=thres, predict_masks=True)
        if contours is not None:
            iou_matrix = pp.compute_iou_matrix_segmentation_contours(contours, pp.convert_mask_to_binary(gt_mask).astype(np.uint8))
            tp, fp, fn, pq, precision, recall, f1_score, mean_iou, dice = \
                pp.compute_metrics_segmentation_from_iou_matrix(iou_matrix=iou_matrix)
        else:
            iou_matrix = pp.compute_iou_matrix_detection(boxes, gt_boxes)
            tp, fp, fn, pq, precision, recall, f1_score, mean_iou, dice = \
                pp.compute_metrics_detection_from_iou_matrix(iou_matrix=iou_matrix)

        print("Threshold", thres)
        print("tp, fp, fn, pq, precision, recall, f1_score, mean_iou:", tp, fp, fn, pq, precision, recall, f1_score, mean_iou)
        pq_data.append((f'{thres:4.2f}', pq, f1_score, precision, recall, mean_iou))


        fig, ax = plt.subplots(1, 1, figsize=(12*4, 12*4), dpi=200)
        plot_boxes(im, gt_boxes, format='yxyx_px', ax=ax, color='blue')
        plot_boxes(im, boxes, format='yxyx_px', ax=ax, show_image=False, color='red')
        plot_dir = results_dir / 'plots' / 'MultiOrg_test_added_eval'
        plot_dir.mkdir(exist_ok=True)
        plt.savefig(plot_dir / f'{str(ds)}_{ds.split}_{idx}_thres_{int(thres*100)}.png', dpi=200)
        plt.close('all')

    # fig, ax = plt.subplots(1, 1, figsize=(10, 10), dpi=80)
    # plot_boxes(im, gt_boxes, format='yxyx_px', ax=ax, color='blue')
    # plot_boxes(im, boxes, format='yxyx_px', ax=ax, show_image=False, color='red')

pq_data = pd.DataFrame(data=pq_data, columns=["thres", "pq", "f1_score", "precision", "recall", "iou"])
pq_data.to_csv(results_dir / f'test_metrics_added_eval_{str(ds)}_{ds.split}.csv', index=False)

pq_data.groupby('thres').mean()
# plot_boxes(im, boxes, format='yxyx_px')

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


976
Threshold 0.5
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 64 107 116 0.2758711760212687 0.3742690058479532 0.35555555555555557 0.36467236467236464 0.7564904904958228
Threshold 0.6
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 61 89 119 0.2815202601479259 0.4066666666666667 0.3388888888888889 0.36969696969696975 0.76148922826898
Threshold 0.7
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 59 71 121 0.2911250693925507 0.45384615384615384 0.3277777777777778 0.3806451612903226 0.7648200975567009
Threshold 0.8
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 55 52 125 0.2994225568747437 0.514018691588785 0.3055555555555556 0.3832752613240418 0.7812206711186493
Threshold 0.85
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 50 44 130 0.2875695909761604 0.5319148936170213 0.2777777777777778 0.36496350364963503 0.7879406792746795
Threshold 0.9
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 43 36 137 0.26428264165667786 0.5443037974683544 0.23

,pq,f1_score,precision,recall,iou
thres,,,,,
0.50,0.487192,0.597299,0.491087,0.824000,0.811195
0.60,0.515346,0.631170,0.538544,0.815637,0.812273
0.70,0.539043,0.659505,0.588395,0.803088,0.813628
0.80,0.558661,0.679730,0.636142,0.775891,0.818681
0.85,0.566661,0.686497,0.666754,0.754057,0.821975
0.90,0.580986,0.704416,0.720046,0.728205,0.821712
0.95,0.582469,0.704343,0.825306,0.651209,0.825612


## Tellu

In [ ]:
ds = dl.Tellu(split='test')
pq_data = []
for idx in range(len(ds)):
    im, gt_mask, gt_boxes, im_path, im_ID = ds[idx]
    im, flatfield = dl.normalize(im)

    H, W = im.shape[:2]
    # patch_size = int(np.ceil(max(H, W) * 7 / 12))
    patch_size = 512
    print(patch_size)

    contours, boxes, scores = samos.forward(im, patch_size=patch_size, predict_masks=True)
    for thres in [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95]:
        contours, boxes, scores = samos.set_threshold(conf_thres=thres, predict_masks=True)
        if contours is not None:
            iou_matrix = pp.compute_iou_matrix_segmentation_contours(contours, pp.convert_mask_to_binary(gt_mask).astype(np.uint8))
            tp, fp, fn, pq, precision, recall, f1_score, mean_iou, dice = \
                pp.compute_metrics_segmentation_from_iou_matrix(iou_matrix=iou_matrix)
        else:
            iou_matrix = pp.compute_iou_matrix_detection(boxes, gt_boxes)
            tp, fp, fn, pq, precision, recall, f1_score, mean_iou, dice = \
                pp.compute_metrics_detection_from_iou_matrix(iou_matrix=iou_matrix)

        print("Threshold", thres)
        print("tp, fp, fn, pq, precision, recall, f1_score, mean_iou:", tp, fp, fn, pq, precision, recall, f1_score, mean_iou)
        pq_data.append((f'{thres:4.2f}', pq, f1_score, precision, recall, mean_iou))


        fig, ax = plt.subplots(1, 1, figsize=(12*4, 12*4), dpi=200)
        plot_boxes(im, gt_boxes, format='yxyx_px', ax=ax, color='blue')
        plot_boxes(im, boxes, format='yxyx_px', ax=ax, show_image=False, color='red')
        plot_dir = results_dir / 'plots' / 'MultiOrg_test_added_eval'
        plot_dir.mkdir(exist_ok=True)
        plt.savefig(plot_dir / f'{str(ds)}_{ds.split}_{idx}_thres_{int(thres*100)}.png', dpi=200)
        plt.close('all')

    # fig, ax = plt.subplots(1, 1, figsize=(10, 10), dpi=80)
    # plot_boxes(im, gt_boxes, format='yxyx_px', ax=ax, color='blue')
    # plot_boxes(im, boxes, format='yxyx_px', ax=ax, show_image=False, color='red')

pq_data = pd.DataFrame(data=pq_data, columns=["thres", "pq", "f1_score", "precision", "recall", "iou"])
pq_data.to_csv(results_dir / f'test_metrics_added_eval_{str(ds)}_{ds.split}.csv', index=False)

pq_data.groupby('thres').mean()
# plot_boxes(im, boxes, format='yxyx_px')

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


976
Threshold 0.5
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 64 107 116 0.2758711760212687 0.3742690058479532 0.35555555555555557 0.36467236467236464 0.7564904904958228
Threshold 0.6
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 61 89 119 0.2815202601479259 0.4066666666666667 0.3388888888888889 0.36969696969696975 0.76148922826898
Threshold 0.7
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 59 71 121 0.2911250693925507 0.45384615384615384 0.3277777777777778 0.3806451612903226 0.7648200975567009
Threshold 0.8
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 55 52 125 0.2994225568747437 0.514018691588785 0.3055555555555556 0.3832752613240418 0.7812206711186493
Threshold 0.85
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 50 44 130 0.2875695909761604 0.5319148936170213 0.2777777777777778 0.36496350364963503 0.7879406792746795
Threshold 0.9
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 43 36 137 0.26428264165667786 0.5443037974683544 0.23

,pq,f1_score,precision,recall,iou
thres,,,,,
0.50,0.487192,0.597299,0.491087,0.824000,0.811195
0.60,0.515346,0.631170,0.538544,0.815637,0.812273
0.70,0.539043,0.659505,0.588395,0.803088,0.813628
0.80,0.558661,0.679730,0.636142,0.775891,0.818681
0.85,0.566661,0.686497,0.666754,0.754057,0.821975
0.90,0.580986,0.704416,0.720046,0.728205,0.821712
0.95,0.582469,0.704343,0.825306,0.651209,0.825612


## MultiOrg

In [ ]:
ds = dl.MultiOrg(split='test_normal')
pq_data = []
for idx in range(len(ds)):
    im, gt_mask, gt_boxes, im_path, im_ID = ds[idx]
    # im, flatfield = dl.normalize(im)
    im = (im / im.max() * 255).astype(np.uint8)
    im = np.stack((im, im, im), axis=2)

    H, W = im.shape[:2]
    patch_size = 512
    print(patch_size)

    contours, boxes, scores = samos.forward(im, patch_size=patch_size, predict_masks=True)
    for thres in [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95]:
        contours, boxes, scores = samos.set_threshold(conf_thres=thres, predict_masks=True)
        if contours is not None:
            iou_matrix = pp.compute_iou_matrix_segmentation_contours(contours, pp.convert_mask_to_binary(gt_mask).astype(np.uint8))
            tp, fp, fn, pq, precision, recall, f1_score, mean_iou, dice = \
                pp.compute_metrics_segmentation_from_iou_matrix(iou_matrix=iou_matrix)
        else:
            iou_matrix = pp.compute_iou_matrix_detection(boxes, gt_boxes)
            tp, fp, fn, pq, precision, recall, f1_score, mean_iou, dice = \
                pp.compute_metrics_detection_from_iou_matrix(iou_matrix=iou_matrix)

        print("Threshold", thres)
        print("tp, fp, fn, pq, precision, recall, f1_score, mean_iou:", tp, fp, fn, pq, precision, recall, f1_score, mean_iou)
        pq_data.append((f'{thres:4.2f}', pq, f1_score, precision, recall, mean_iou))


        fig, ax = plt.subplots(1, 1, figsize=(12*4, 12*4), dpi=200)
        plot_boxes(im, gt_boxes, format='yxyx_px', ax=ax, color='blue')
        plot_boxes(im, boxes, format='yxyx_px', ax=ax, show_image=False, color='red')
        plot_dir = results_dir / 'plots' / 'MultiOrg_test_added_eval'
        plot_dir.mkdir(exist_ok=True)
        plt.savefig(plot_dir / f'{str(ds)}_{ds.split}_{idx}_thres_{int(thres*100)}.png', dpi=200)
        plt.close('all')

    # fig, ax = plt.subplots(1, 1, figsize=(10, 10), dpi=80)
    # plot_boxes(im, gt_boxes, format='yxyx_px', ax=ax, color='blue')
    # plot_boxes(im, boxes, format='yxyx_px', ax=ax, show_image=False, color='red')

pq_data = pd.DataFrame(data=pq_data, columns=["thres", "pq", "f1_score", "precision", "recall", "iou"])
pq_data.to_csv(results_dir / f'test_metrics_added_eval_{str(ds)}_{ds.split}.csv', index=False)

pq_data.groupby('thres').mean()
# plot_boxes(im, boxes, format='yxyx_px')

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


976
Threshold 0.5
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 64 107 116 0.2758711760212687 0.3742690058479532 0.35555555555555557 0.36467236467236464 0.7564904904958228
Threshold 0.6
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 61 89 119 0.2815202601479259 0.4066666666666667 0.3388888888888889 0.36969696969696975 0.76148922826898
Threshold 0.7
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 59 71 121 0.2911250693925507 0.45384615384615384 0.3277777777777778 0.3806451612903226 0.7648200975567009
Threshold 0.8
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 55 52 125 0.2994225568747437 0.514018691588785 0.3055555555555556 0.3832752613240418 0.7812206711186493
Threshold 0.85
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 50 44 130 0.2875695909761604 0.5319148936170213 0.2777777777777778 0.36496350364963503 0.7879406792746795
Threshold 0.9
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 43 36 137 0.26428264165667786 0.5443037974683544 0.23

,pq,f1_score,precision,recall,iou
thres,,,,,
0.50,0.487192,0.597299,0.491087,0.824000,0.811195
0.60,0.515346,0.631170,0.538544,0.815637,0.812273
0.70,0.539043,0.659505,0.588395,0.803088,0.813628
0.80,0.558661,0.679730,0.636142,0.775891,0.818681
0.85,0.566661,0.686497,0.666754,0.754057,0.821975
0.90,0.580986,0.704416,0.720046,0.728205,0.821712
0.95,0.582469,0.704343,0.825306,0.651209,0.825612


In [ ]:
ds = dl.MultiOrg(split='test_macros')
pq_data = []
for idx in range(len(ds)):
    im, gt_mask, gt_boxes, im_path, im_ID = ds[idx]
    # im, flatfield = dl.normalize(im)
    im = (im / im.max() * 255).astype(np.uint8)
    im = np.stack((im, im, im), axis=2)

    H, W = im.shape[:2]
    patch_size = 512
    print(patch_size)

    contours, boxes, scores = samos.forward(im, patch_size=patch_size, predict_masks=True)
    for thres in [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95]:
        contours, boxes, scores = samos.set_threshold(conf_thres=thres, predict_masks=True)
        if contours is not None:
            iou_matrix = pp.compute_iou_matrix_segmentation_contours(contours, pp.convert_mask_to_binary(gt_mask).astype(np.uint8))
            tp, fp, fn, pq, precision, recall, f1_score, mean_iou, dice = \
                pp.compute_metrics_segmentation_from_iou_matrix(iou_matrix=iou_matrix)
        else:
            iou_matrix = pp.compute_iou_matrix_detection(boxes, gt_boxes)
            tp, fp, fn, pq, precision, recall, f1_score, mean_iou, dice = \
                pp.compute_metrics_detection_from_iou_matrix(iou_matrix=iou_matrix)

        print("Threshold", thres)
        print("tp, fp, fn, pq, precision, recall, f1_score, mean_iou:", tp, fp, fn, pq, precision, recall, f1_score, mean_iou)
        pq_data.append((f'{thres:4.2f}', pq, f1_score, precision, recall, mean_iou))


        fig, ax = plt.subplots(1, 1, figsize=(12*4, 12*4), dpi=200)
        plot_boxes(im, gt_boxes, format='yxyx_px', ax=ax, color='blue')
        plot_boxes(im, boxes, format='yxyx_px', ax=ax, show_image=False, color='red')
        plot_dir = results_dir / 'plots' / 'MultiOrg_test_added_eval'
        plot_dir.mkdir(exist_ok=True)
        plt.savefig(plot_dir / f'{str(ds)}_{ds.split}_{idx}_thres_{int(thres*100)}.png', dpi=200)
        plt.close('all')

    # fig, ax = plt.subplots(1, 1, figsize=(10, 10), dpi=80)
    # plot_boxes(im, gt_boxes, format='yxyx_px', ax=ax, color='blue')
    # plot_boxes(im, boxes, format='yxyx_px', ax=ax, show_image=False, color='red')

pq_data = pd.DataFrame(data=pq_data, columns=["thres", "pq", "f1_score", "precision", "recall", "iou"])
pq_data.to_csv(results_dir / f'test_metrics_added_eval_{str(ds)}_{ds.split}.csv', index=False)

pq_data.groupby('thres').mean()
# plot_boxes(im, boxes, format='yxyx_px')

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


976
Threshold 0.5
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 64 107 116 0.2758711760212687 0.3742690058479532 0.35555555555555557 0.36467236467236464 0.7564904904958228
Threshold 0.6
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 61 89 119 0.2815202601479259 0.4066666666666667 0.3388888888888889 0.36969696969696975 0.76148922826898
Threshold 0.7
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 59 71 121 0.2911250693925507 0.45384615384615384 0.3277777777777778 0.3806451612903226 0.7648200975567009
Threshold 0.8
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 55 52 125 0.2994225568747437 0.514018691588785 0.3055555555555556 0.3832752613240418 0.7812206711186493
Threshold 0.85
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 50 44 130 0.2875695909761604 0.5319148936170213 0.2777777777777778 0.36496350364963503 0.7879406792746795
Threshold 0.9
tp, fp, fn, pq, precision, recall, f1_score, mean_iou: 43 36 137 0.26428264165667786 0.5443037974683544 0.23

,pq,f1_score,precision,recall,iou
thres,,,,,
0.50,0.487192,0.597299,0.491087,0.824000,0.811195
0.60,0.515346,0.631170,0.538544,0.815637,0.812273
0.70,0.539043,0.659505,0.588395,0.803088,0.813628
0.80,0.558661,0.679730,0.636142,0.775891,0.818681
0.85,0.566661,0.686497,0.666754,0.754057,0.821975
0.90,0.580986,0.704416,0.720046,0.728205,0.821712
0.95,0.582469,0.704343,0.825306,0.651209,0.825612


In [4]:
ds = dl.MultiOrg(split='test_normal')

for idx in range(len(ds)):
    if idx != 0:
        break

    im, gt_mask, gt_boxes, im_path, im_ID = ds[idx]
    im = (im / im.max() * 255).astype(np.uint8)
    im = np.stack((im, im, im), axis=2)

    
    masks, boxes, scores = samos.forward(im, patch_size=(300, 1024), predict_masks=True)
    masks, boxes, scores = samos.set_threshold(conf_thres=0.95, predict_masks=True)

    fig, ax = plt.subplots(1, 1, figsize=(12*2, 12*2), dpi=40)
    plot_boxes(im, gt_boxes, format='yxyx_px', ax=ax, color='blue')
    plot_boxes(im, boxes, format='yxyx_px', ax=ax, show_image=False, color='red')
    plot_dir = results_dir / 'plots' / 'MultiOrg_test_added_eval'
    plot_dir.mkdir(exist_ok=True)
    plt.savefig(plot_dir / f'{str(ds)}_{ds.split}_{idx}_thres_95.png', dpi=200)
    plt.close('all')
    
    masks, boxes, scores = samos.set_threshold(conf_thres=0.85, predict_masks=True)
    
    fig, ax = plt.subplots(1, 1, figsize=(12*2, 12*2), dpi=40)
    plot_boxes(im, gt_boxes, format='yxyx_px', ax=ax, color='blue')
    plot_boxes(im, boxes, format='yxyx_px', ax=ax, show_image=False, color='red')
    plot_dir = results_dir / 'plots' / 'MultiOrg_test_added_eval'
    plot_dir.mkdir(exist_ok=True)
    plt.savefig(plot_dir / f'{str(ds)}_{ds.split}_{idx}_thres_85.png', dpi=200)
    plt.close('all')
    
    masks, boxes, scores = samos.set_threshold(conf_thres=0.75, predict_masks=True)
    
    fig, ax = plt.subplots(1, 1, figsize=(12*2, 12*2), dpi=40)
    plot_boxes(im, gt_boxes, format='yxyx_px', ax=ax, color='blue')
    plot_boxes(im, boxes, format='yxyx_px', ax=ax, show_image=False, color='red')
    plot_dir = results_dir / 'plots' / 'MultiOrg_test_added_eval'
    plot_dir.mkdir(exist_ok=True)
    plt.savefig(plot_dir / f'{str(ds)}_{ds.split}_{idx}_thres_75.png', dpi=200)
    plt.close('all')